<a href="https://colab.research.google.com/github/EnricoSouzaAfonso/ImersaoGemini_Alura_Conrad/blob/main/Conrad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install google-genai markdown2 beautifulsoup4 lxml Pygments ipywidgets

In [ ]:
# -- Imports e Configuração Inicial (Modernizado para google-genai v1.x) --
import os
import sys

# Garante acesso ao diretório src/ caso esteja rodando local ou clonado
if "." not in sys.path:
    sys.path.append(".")

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY') or os.environ.get('GEMINI_API_KEY')

if not GOOGLE_API_KEY:
    raise SystemExit("Erro: Chave da API não configurada nos Secrets do Colab ou nas Variáveis de Ambiente.")

model_name_conrad = "gemini-2.5-flash"
print(f"Ambiente configurado com sucesso! Modelo ativo: {model_name_conrad}")

In [ ]:
# -- Célula Utilitária: Listar Modelos Disponíveis com google-genai --
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

print("=" * 70)
print("MODELOS DISPONÍVEIS (suporte a generateContent):")
print("=" * 70)

for model in client.models.list():
    actions = getattr(model, "supported_actions", []) or []
    if "generateContent" in actions:
        print(f"Nome de Exibição: {model.display_name}")
        print(f"  Nome Técnico    : {model.name}")
        print(f"  Entrada Máx     : {getattr(model, 'input_token_limit', 'N/D')}")
        print(f"  Saída Máx       : {getattr(model, 'output_token_limit', 'N/D')}")
        print("-" * 40)

In [ ]:
# -- Célula Principal do Conrad: Interface e Lógica do Chat com IPYWIDGETS --
# Lembre-se de executar a célula com '!pip -q install google-genai markdown2 beautifulsoup4 lxml pygments' ANTES desta.

# Imports e Bibliotecas Necessárias
from IPython.display import display, Markdown, clear_output, Javascript, HTML
from google import genai
from google.genai import types
from src.conrad_core import ConradEngine
import html
import re
import markdown2
from bs4 import BeautifulSoup
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Layout, HTML as IPyHTML
from pygments.lexers import get_all_lexers

# ==============================================================================
# --- MENTE DO CONRAD (SYSTEM INSTRUCTIONS) ---
# ==============================================================================
# A personalidade, as regras e o comportamento do Conrad são definidos aqui.

from src.conrad_persona import SYSTEM_INSTRUCTION_CONRAD, get_system_instruction
system_instruction_conrad = get_system_instruction()

# ==============================================================================
# --- LÓGICA DO CHAT ---
# ==============================================================================

def _build_dynamic_language_map():
    lang_map = {}
    for lexer in get_all_lexers():
        if not lexer[2]: continue
        primeira_extensao = lexer[2][0][1:]
        mime_type = lexer[3][0] if lexer[3] else 'text/plain'
        info = {'ext': primeira_extensao, 'mime': mime_type}
        for alias in lexer[1]:
            lang_map[alias.lower()] = info
    return lang_map

LANG_INFO_MAP = _build_dynamic_language_map()

proximo_id_bloco_codigo = 0

generation_config_conrad = {
    "temperature": 0.7, "top_p": 0.95, "top_k": 40, "candidate_count": 1,
}
safety_settings_conrad = [
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
]

model_conrad = None
chat_session_conrad = None
initialization_error_message = None

MSG_ENCERRAR_PT_UI = "PT-BR: Para encerrar, digite 'fim' e envie."
MSG_ENCERRAR_EN_UI = "ENG: (To end, type 'fim' and send.)"
MSG_RESUMO_PT_UI = "PT-BR: Peça um resumo da conversa a qualquer momento, se desejar."
MSG_RESUMO_EN_UI = "ENG: (Ask for a conversation summary anytime, if you wish.)"
SYS_MSG_MODEL_INIT_FAIL_PT = "PT-BR: Falha ao inicializar o modelo do Conrad. Verifique a API Key, o nome do modelo e se 'system_instruction_conrad' está definida."
SYS_MSG_MODEL_INIT_FAIL_EN = "ENG: Failed to initialize Conrad's model. Check API Key, model name and if 'system_instruction_conrad' is defined."
SYS_MSG_MODEL_NAME_MISSING_PT = "PT-BR: A variável 'model_name_conrad' não foi definida ou está vazia. Execute a célula de configuração."
SYS_MSG_MODEL_NAME_MISSING_EN = "ENG: The 'model_name_conrad' variable is not defined or is empty. Run the configuration cell."
SYS_MSG_SYS_INSTRUCTION_MISSING_PT = "PT-BR: A variável 'system_instruction_conrad' não foi definida. Execute a célula que a define."
SYS_MSG_SYS_INSTRUCTION_MISSING_EN = "ENG: The 'system_instruction_conrad' variable is not defined. Run the cell that defines it."
SYS_MSG_CHAT_INIT_FAIL_PT = "PT-BR: Falha ao iniciar a sessão de chat com o Conrad."
SYS_MSG_CHAT_INIT_FAIL_EN = "ENG: Failed to start chat session with Conrad."
SYS_MSG_CHAR_LIMIT_PT = "PT-BR: Sua mensagem excede o limite de {} caracteres e não foi enviada."
SYS_MSG_CHAR_LIMIT_EN = "ENG: Your message exceeds the {} character limit and was not sent."
SYS_MSG_NO_CHAT_SESSION_PT = "PT-BR: Erro: Sessão de chat não iniciada."
SYS_MSG_NO_CHAT_SESSION_EN = "ENG: Error: Chat session not started."
SYS_MSG_API_ERROR_WRAPPER_PT = "PT-BR: Erro ao comunicar com o modelo: `{error_detail}`"
SYS_MSG_API_ERROR_WRAPPER_EN = "ENG: Error communicating with the model: `{error_detail}`"
SYS_MSG_FAREWELL_ERROR_WRAPPER_PT = "PT-BR: Erro ao tentar obter despedida do Conrad: `{error_detail}`"
SYS_MSG_FAREWELL_ERROR_WRAPPER_EN = "ENG: Error trying to get Conrad's farewell: `{error_detail}`"
SYS_MSG_CRITICAL_FAILURE_PT = "PT-BR: Falha crítica na inicialização da interface."
SYS_MSG_CRITICAL_FAILURE_EN = "ENG: Critical interface initialization failure."
SYS_MSG_UNKNOWN_INIT_ERROR_PT = "PT-BR: Erro desconhecido durante a inicialização."
SYS_MSG_UNKNOWN_INIT_ERROR_EN = "ENG: Unknown error during initialization."

_system_instruction_to_use = None
if 'system_instruction_conrad' not in globals() or not system_instruction_conrad or "--- (SEU PROMPT COMPLETO" in system_instruction_conrad:
    initialization_error_message = f"{SYS_MSG_SYS_INSTRUCTION_MISSING_PT}<br>{SYS_MSG_SYS_INSTRUCTION_MISSING_EN}"
else:
    _system_instruction_to_use = system_instruction_conrad

if 'model_name_conrad' not in globals() or not model_name_conrad:
    err_msg_model_name = f"{SYS_MSG_MODEL_NAME_MISSING_PT}<br>{SYS_MSG_MODEL_NAME_MISSING_EN}"
    initialization_error_message = f"{initialization_error_message}<br><br>{err_msg_model_name}" if initialization_error_message else err_msg_model_name
    model_name_conrad = "gemini-2.5-flash" # Fallback
    print(f"PT-BR: Usando modelo de fallback: {model_name_conrad}.\nENG: Using fallback model: {model_name_conrad}.")

if not initialization_error_message:
    try:
        model_conrad = ConradEngine(
            api_key=GOOGLE_API_KEY if 'GOOGLE_API_KEY' in globals() else None,
            model_name=model_name_conrad,
            system_instruction=_system_instruction_to_use
        )
    except Exception as e:
        err_detail = html.escape(str(e))
        pt_msg = SYS_MSG_MODEL_INIT_FAIL_PT
        en_msg = SYS_MSG_MODEL_INIT_FAIL_EN
        initialization_error_message = f"{pt_msg}<br>{en_msg}<br><br>PT-BR: Detalhe: {err_detail}<br>ENG: Detail: {err_detail}"
        model_conrad = None

CONRAD_THINKING_INDICATOR = "🤔💭"
MAX_USER_INPUT_LENGTH = 20000
MODEL_CONTEXT_TOKEN_LIMIT = 1048576

chat_history_initial_layout = Layout(width='calc(100% - 20px)', height='0px', min_height='0px', overflow_y='auto', border='none', padding='0px', margin='5px 0px')
chat_history_visible_layout = Layout(width='calc(100% - 2px)', max_height='450px', overflow_y='auto', border='1px solid #444', padding='10px 0px', margin='5px 0px')
input_box_layout = Layout(width='100%', margin='10px 0 0 0', align_items='center')
input_layout_widget = Layout(width='70%', min_width='200px', margin='0px 5px 0px 0px')
button_layout_widget = Layout(width='auto', min_width='80px')
status_layout_widget = Layout(margin='0px 0px 5px 0px', width='auto')

chat_history_output = widgets.Output(layout=chat_history_initial_layout)
chat_history_output.add_class("conrad-chat-history")
user_input_area = widgets.Textarea(layout=input_layout_widget, rows=2, disabled=True)
send_button = widgets.Button(description='Enviar', button_style='primary', tooltip='Enviar mensagem', layout=button_layout_widget, disabled=True)
user_input_area.add_class("conrad-user-input-textarea")
send_button.add_class("conrad-send-button")
status_message_area = IPyHTML(value="", layout=status_layout_widget)
input_box = HBox([user_input_area, send_button], layout=input_box_layout)
instrucoes_html_valor = f"""
<div style='font-size:small; color:#fff; font-weight:bold; padding-bottom: 5px;'>
    {MSG_ENCERRAR_PT_UI}<br>
    {MSG_ENCERRAR_EN_UI}<br>
    {MSG_RESUMO_PT_UI}<br>
    {MSG_RESUMO_EN_UI}
</div>
"""
instrucoes_widget = IPyHTML(value=instrucoes_html_valor)
token_count_display_widget = IPyHTML(value="", layout=Layout(margin='5px 0 8px 0'))
ui_container = VBox([
    instrucoes_widget,
    token_count_display_widget,
    chat_history_output,
    status_message_area,
    input_box
], layout=Layout(width='100%', max_width='1050px', padding='10px', border='1px solid #333', background_color='#222', overflow_x='hidden'))

conversa_encerrada_flag = False
_last_message_was_thinking_indicator = False
_chat_history_initialized = False
displayed_chat_messages_list = []

def _update_token_count_display():
    if chat_session_conrad and model_conrad:
        try:
            current_tokens = 0
            limit_to_display = MODEL_CONTEXT_TOKEN_LIMIT
            current_tokens = model_conrad.count_tokens()
            warning_color = "#ccc"
            percentage_used = (current_tokens / limit_to_display) * 100 if limit_to_display > 0 else 0
            if percentage_used > 75: warning_color = "orange"
            if percentage_used > 90: warning_color = "red"
            tt_pt = "PT-BR: Olá! Sou o Conrad. 😊 Tokens são 'blocos de construção' que uso para entender e responder. Esta contagem mostra os tokens usados e o limite. **Importante:** Se o limite for atingido, posso 'esquecer' o início da conversa."
            tt_en = "ENG: Hi! I'm Conrad. 😊 Tokens are 'building blocks' I use to understand and reply. This count shows tokens used and the total limit. **Important:** If the limit is reached, I might 'forget' the start of our chat."
            full_tooltip_text = f"{tt_pt}\n\n{tt_en}"
            svg = """<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" width="1.0em" height="1.0em" fill="currentColor" style="vertical-align: -0.12em;"><path d="M0 0h24v24H0z" fill="none"/><path d="M11 7h2v2h-2zm0 4h2v6h-2zm1-9C6.48 2 2 6.48 2 12s4.48 10 10 10 10-4.48 10-10S17.52 2 12 2zm0 18c-4.41 0-8-3.59-8-8s3.59-8 8-8 8 3.59 8 8-3.59 8-8 8z"/></svg>"""
            info_icon = f"<span title='{html.escape(full_tooltip_text, quote=True)}' style='cursor:default; margin-left: 4px; display: inline-flex; align-items: center;'>{' '.join(svg.split())}</span>"
            token_count_display_widget.value = f"<div style='font-size:small; color:{warning_color}; text-align:right; font-weight:bold;'>Tokens: {current_tokens:,} / {limit_to_display:,}{info_icon}</div>"
        except Exception:
            token_count_display_widget.value = "<div style='font-size:small; color:orange; text-align:right; font-weight:bold;'>Erro ao buscar tokens</div>"
    else:
        tt_pt = "PT-BR: Olá! Sou o Conrad. 😊 Tokens são 'blocos de construção' que uso para entender e responder. Esta contagem mostra os tokens usados e o limite. **Importante:** Se o limite for atingido, posso 'esquecer' o início da conversa."
        tt_en = "ENG: Hi! I'm Conrad. 😊 Tokens are 'building blocks' I use to understand and reply. This count shows tokens used and the total limit. **Important:** If the limit is reached, I might 'forget' the start of our chat."
        full_tooltip_text = f"{tt_pt}\n\n{tt_en}"
        svg = """<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24" width="1.0em" height="1.0em" fill="currentColor" style="vertical-align: -0.12em;"><path d="M0 0h24v24H0z" fill="none"/><path d="M11 7h2v2h-2zm0 4h2v6h-2zm1-9C6.48 2 2 6.48 2 12s4.48 10 10 10 10-4.48 10-10S17.52 2 12 2zm0 18c-4.41 0-8-3.59-8-8s3.59-8 8-8 8 3.59 8 8-3.59 8-8 8z"/></svg>"""
        info_icon = f"<span title='{html.escape(full_tooltip_text, quote=True)}' style='cursor:default; margin-left: 4px; display: inline-flex; align-items: center;'>{' '.join(svg.split())}</span>"
        token_count_display_widget.value = f"<div style='font-size:small; color:#ccc; text-align:right; font-weight:bold;'>Tokens: 0 / {MODEL_CONTEXT_TOKEN_LIMIT:,}{info_icon}</div>"

def _reset_chat_display_state():
    global displayed_chat_messages_list, _last_message_was_thinking_indicator, _chat_history_initialized, proximo_id_bloco_codigo
    displayed_chat_messages_list = []
    _last_message_was_thinking_indicator = False
    _chat_history_initialized = False
    proximo_id_bloco_codigo = 0
    if chat_history_output:
        with chat_history_output: clear_output(wait=False)
        chat_history_output.layout = chat_history_initial_layout
    _update_token_count_display()

def _render_chat_history():
    global _chat_history_initialized
    if chat_history_output:
        with chat_history_output: clear_output(wait=True)
    if not displayed_chat_messages_list:
        if _chat_history_initialized:
            if chat_history_output: chat_history_output.layout = chat_history_initial_layout
            _chat_history_initialized = False
        return
    if not _chat_history_initialized:
        if chat_history_output: chat_history_output.layout = chat_history_visible_layout
        _chat_history_initialized = True
    full_html_content_str = "".join("<hr style='border-color:#ddd; margin-top: 5px; margin-bottom: 5px; width: 100%; box-sizing: border-box;'>" if msg_str == "---" else msg_str for msg_str in displayed_chat_messages_list)
    if chat_history_output:
        with chat_history_output:
            display(HTML(full_html_content_str))
            scroll_js = """(function() { var ch = document.querySelector('.conrad-chat-history'); if (ch) { var sa = ch.querySelector('.jp-OutputArea-output') || ch; setTimeout(function() { if (sa) sa.scrollTop = sa.scrollHeight; }, 150); } })();"""
            display(Javascript(scroll_js))

def _extrair_traducao_copiado(texto_markdown):
    padrao = r"\[TRADUCAO_COPIADO:\s*(.+?)\s*\]"
    match = re.search(padrao, texto_markdown)
    if match:
        traducao = match.group(1)
        return traducao, texto_markdown
    return "Código copiado com sucesso!", texto_markdown

def _extrair_todos_os_blocos_de_codigo_com_linguagem(texto_markdown_completo):
    blocos_encontrados = []
    padrao_bloco_codigo = r"^\s*```\s*([^\r\n]*)\r?\n(.*?)\r?\n^\s*```"

    for match in re.finditer(padrao_bloco_codigo, texto_markdown_completo, re.MULTILINE | re.DOTALL):
        linguagem_bruta = match.group(1)
        linguagem_final = linguagem_bruta.lower().strip() if linguagem_bruta else None

        palavras_a_ignorar_como_linguagem = ["text", "code", "plain", "plaintext", ""]
        if linguagem_final in palavras_a_ignorar_como_linguagem:
            linguagem_final = None
        elif linguagem_final and (linguagem_final.startswith("#") or linguagem_final.startswith("//") or linguagem_final.startswith("/*")):
            linguagem_final = None

        blocos_encontrados.append(linguagem_final)

    return blocos_encontrados

def _get_file_info(language):
    global LANG_INFO_MAP
    default_info = {'ext': '.txt', 'mime': 'text/plain;charset=utf-8'}
    if language:
        lang_info = LANG_INFO_MAP.get(language.lower(), default_info)

        if lang_info['mime'].startswith('text/') and 'charset' not in lang_info['mime']:
            lang_info['mime'] += ';charset=utf-8'
        return lang_info
    return default_info

SVG_CARTOLA_CONRAD = """
<svg class="conrad-avatar-icon" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 100 85" width="22px" height="22px">
  <path fill="black" d="M5,65 Q5,50 15,50 L85,50 Q95,50 95,65 L95,70 L5,70 L5,65 Z" />
  <rect x="25" y="0" width="50" height="55" fill="black" />
  <rect x="20" y="50" width="60" height="7" fill="white" />
</svg>
"""

def _adicionar_mensagem_ao_historico_logica(quem, texto_markdown_original, is_thinking_indicator=False):
    global _last_message_was_thinking_indicator, displayed_chat_messages_list, proximo_id_bloco_codigo
    html_final_para_lista = ""

    if is_thinking_indicator:
        html_final_para_lista = f"<div class='conrad-response-item' style='padding: 0 10px;'><div class='conrad-response-header'><span class='conrad-avatar'>{SVG_CARTOLA_CONRAD}</span><b>Conrad:</b><span style='margin-left: 5px;'>{CONRAD_THINKING_INDICATOR}</span></div></div>"
        if _last_message_was_thinking_indicator and displayed_chat_messages_list:
            displayed_chat_messages_list.pop()
        _last_message_was_thinking_indicator = True
    else:
        if _last_message_was_thinking_indicator and displayed_chat_messages_list:
            displayed_chat_messages_list.pop()
        _last_message_was_thinking_indicator = False

        traducao_para_popup = "Código copiado com sucesso!"
        texto_markdown_para_html = texto_markdown_original

        linguagens_dos_blocos_markdown = []
        badge_default_color = {'bg': '#4a4a4a', 'text': '#ffffff'}

        if quem == "Conrad":
            traducao_para_popup, _ = _extrair_traducao_copiado(texto_markdown_original)
            padrao_tag_traducao = r"\[TRADUCAO_COPIADO:\s*.*?\]"
            texto_limpo_de_tags_de_traducao = re.sub(padrao_tag_traducao, "", texto_markdown_original).strip()
            linguagens_dos_blocos_markdown = _extrair_todos_os_blocos_de_codigo_com_linguagem(texto_limpo_de_tags_de_traducao)
            texto_markdown_para_html = texto_limpo_de_tags_de_traducao

        if quem == "Você":
            texto_escapado = html.escape(texto_markdown_para_html).replace('\n', '<br>\n')
            html_final_para_lista = f"<div class='chat-message-user-text' style='padding: 0 10px;'><b>{quem}:</b> {texto_escapado}</div>"
        elif quem == "Sistema":
            texto_html_sistema = texto_markdown_para_html.replace('\n', '<br>')
            html_final_para_lista = f"<div class='chat-message-system-text' style='color:red; font-style:italic; padding: 0 10px;'><div class='conrad-response-header'><b>{quem}:</b></div><div class='conrad-response-content'>{texto_html_sistema}</div></div>"
        else: # Conrad
            html_gerado = markdown2.markdown(
                texto_markdown_para_html,
                extras=["fenced-code-blocks", "code-friendly", "tables", "smarty-pants", "code-ids"]
            )

            soup = BeautifulSoup(html_gerado, 'lxml')
            code_containers_html = soup.find_all('div', class_='codehilite')

            svg_icon_copiar = """<svg xmlns="http://www.w3.org/2000/svg" height="16px" viewBox="0 0 24 24" width="16px"><path d="M0 0h24v24H0V0z" fill="none"/><path d="M16 1H4c-1.1 0-2 .9-2 2v14h2V3h12V1zm3 4H8c-1.1 0-2 .9-2 2v14c0 1.1.9 2 2 2h11c1.1 0 2-.9 2-2V7c0-1.1-.9-2-2-2zm0 16H8V7h11v14z"/></svg>"""
            svg_icon_download = """<svg xmlns="http://www.w3.org/2000/svg" height="16px" viewBox="0 0 24 24" width="16px"><path d="M0 0h24v24H0V0z" fill="none"/><path d="M19 9h-4V3H9v6H5l7 7 7-7zm-8 2V5h2v6h1.17L12 13.17 9.83 11H11zm-6 7h14v2H5v-2z"/></svg>"""

            for idx, container_html in enumerate(code_containers_html):
                pre_tag = container_html.find('pre')
                if pre_tag:
                    code_block_id = pre_tag.get('id')
                    if not code_block_id:
                        code_block_id = f"conrad-code-{proximo_id_bloco_codigo}"
                        pre_tag['id'] = code_block_id

                    popup_id = f"conrad-popup-{proximo_id_bloco_codigo}"

                    linguagem_para_este_bloco = None
                    if idx < len(linguagens_dos_blocos_markdown):
                        linguagem_para_este_bloco = linguagens_dos_blocos_markdown[idx]

                    file_info = _get_file_info(linguagem_para_este_bloco)
                    nome_arquivo_sugerido = f"conrad_codigo_{proximo_id_bloco_codigo}{file_info['ext']}"
                    mime_type_sugerido = file_info['mime']

                    data_msg_copiado_escaped = html.escape(traducao_para_popup, quote=True)

                    botoes_container_html = f"""<div class="code-buttons-container"><button onclick="copiarCodigoParaClipboard('{code_block_id}', this, '{popup_id}')" class="botao-code-action botao-copiar-codigo" title="Copiar código" data-msg-copiado="{data_msg_copiado_escaped}">{svg_icon_copiar}</button><button onclick="baixarCodigoComoArquivo('{code_block_id}', '{nome_arquivo_sugerido}', '{mime_type_sugerido}')" class="botao-code-action botao-download-codigo" title="Baixar código">{svg_icon_download}</button><span id="{popup_id}" class="copiado-popup"></span></div>"""
                    botoes_container_tag = BeautifulSoup(botoes_container_html, 'html.parser').div
                    container_html.insert(0, botoes_container_tag)

                    if linguagem_para_este_bloco:
                        indicator_html = f'<span class="code-language-indicator" style="background-color: {badge_default_color["bg"]}; color: {badge_default_color["text"]};">{html.escape(linguagem_para_este_bloco)}</span>'
                        indicator_tag = BeautifulSoup(indicator_html, 'html.parser').span
                        container_html.append(indicator_tag)

                    proximo_id_bloco_codigo += 1

            html_do_conteudo = str(soup)
            html_final_para_lista = f"<div class='conrad-response-item' style='padding: 0 10px;'><div class='conrad-response-header'><span class='conrad-avatar'>{SVG_CARTOLA_CONRAD}</span><b>{quem}:</b></div><div class='conrad-response-content'>{html_do_conteudo}</div></div>"

    displayed_chat_messages_list.append(html_final_para_lista)
    if quem != "Sistema" and not is_thinking_indicator:
         displayed_chat_messages_list.append("---")
    _render_chat_history()

def _adicionar_mensagem_ao_historico(quem, texto, is_thinking=False):
    _adicionar_mensagem_ao_historico_logica(quem, texto, is_thinking_indicator=is_thinking)

def _focus_user_input_area():
    display(Javascript("setTimeout(function() { const ta = document.querySelector('.conrad-user-input-textarea textarea'); if (ta && !ta.disabled) { ta.focus(); } }, 50);"))

def lidar_com_envio_do_usuario(prompt_usuario):
    global conversa_encerrada_flag, chat_session_conrad
    if not prompt_usuario: return
    if len(prompt_usuario) > MAX_USER_INPUT_LENGTH:
        msg_pt, msg_en = SYS_MSG_CHAR_LIMIT_PT.format(MAX_USER_INPUT_LENGTH), SYS_MSG_CHAR_LIMIT_EN.format(MAX_USER_INPUT_LENGTH)
        status_message_area.value = f"<p style='color:orange; font-size:small;'>{msg_pt}<br>{msg_en}</p>"
        if not conversa_encerrada_flag: send_button.disabled = user_input_area.disabled = False; _focus_user_input_area()
        return
    if not chat_session_conrad: _adicionar_mensagem_ao_historico("Sistema", f"{SYS_MSG_NO_CHAT_SESSION_PT}<br>{SYS_MSG_NO_CHAT_SESSION_EN}"); return

    prompt_strip = prompt_usuario.strip()
    if prompt_strip == "fim":
        conversa_encerrada_flag = True; _adicionar_mensagem_ao_historico("Você", prompt_usuario)
        if chat_session_conrad:
            # Também adicionamos a defesa aqui para a mensagem de despedida
            try:
                resposta_final = chat_session_conrad.send_message(prompt_usuario)
                if getattr(resposta_final, "text", None):
                    _adicionar_mensagem_ao_historico("Conrad", resposta_final.text)
                else:
                    _adicionar_mensagem_ao_historico("Conrad", "(Não foi possível gerar uma despedida, mas a sessão foi encerrada.)")
            except Exception as e: _adicionar_mensagem_ao_historico("Sistema", f"{SYS_MSG_FAREWELL_ERROR_WRAPPER_PT.format(error_detail=html.escape(str(e)))}<br>{SYS_MSG_FAREWELL_ERROR_WRAPPER_EN.format(error_detail=html.escape(str(e)))}")
        _update_token_count_display(); status_message_area.value = "<p style='color:lightgreen;'>Conversa encerrada.</p>"
        user_input_area.disabled = send_button.disabled = True; input_box.layout.display = 'none'
        return

    _adicionar_mensagem_ao_historico("Você", prompt_usuario)
    if conversa_encerrada_flag: return
    send_button.disabled = True; status_message_area.value = ""
    _adicionar_mensagem_ao_historico("Conrad", CONRAD_THINKING_INDICATOR, is_thinking=True)

    # <-- INÍCIO DA CORREÇÃO APLICADA -->
    try:
        resposta_completa = chat_session_conrad.send_message(prompt_usuario)

        if getattr(resposta_completa, "text", None):
            texto_da_resposta = resposta_completa.text
            _adicionar_mensagem_ao_historico("Conrad", texto_da_resposta)
        else:
            if _last_message_was_thinking_indicator and displayed_chat_messages_list:
                displayed_chat_messages_list.pop()
                _last_message_was_thinking_indicator = False

            block_reason = "Não especificado"
            if getattr(resposta_completa, "prompt_feedback", None):
                block_reason = str(getattr(resposta_completa.prompt_feedback, "block_reason", block_reason))

            error_message_pt = f"A resposta do Conrad foi bloqueada pelo filtro de segurança. Motivo: {block_reason}. Por favor, tente reformular sua pergunta."
            error_message_en = f"Conrad's response was blocked by the safety filter. Reason: {block_reason}. Please try rephrasing your question."
            _adicionar_mensagem_ao_historico("Sistema", f"{error_message_pt}<br>{error_message_en}")

    except Exception as e:
        err = html.escape(str(e)); msg_pt, msg_en = SYS_MSG_API_ERROR_WRAPPER_PT.format(error_detail=err), SYS_MSG_API_ERROR_WRAPPER_EN.format(error_detail=err)
        _adicionar_mensagem_ao_historico("Sistema", f"{msg_pt}<br>{msg_en}")
        status_message_area.value = f"<p style='color:red; font-size:small;'>{msg_pt}<br>{msg_en}</p>"
    finally:
        if not conversa_encerrada_flag: send_button.disabled = user_input_area.disabled = False; _focus_user_input_area()
        _update_token_count_display()
    # <-- FIM DA CORREÇÃO APLICADA -->


def on_send_button_clicked(b):
    prompt = user_input_area.value.strip()
    if prompt:
        user_input_area.value = ""
        display(Javascript("const ta = document.querySelector('.conrad-user-input-textarea textarea'); if (ta) { if (typeof ta.adjustTextareaHeight === 'function') ta.adjustTextareaHeight(); if (!ta.disabled) ta.focus(); }"))
        lidar_com_envio_do_usuario(prompt)

from google.colab import output as colab_output
colab_output.register_callback('notebook.handleConradUserSubmitFromEnter', lidar_com_envio_do_usuario)
send_button.on_click(on_send_button_clicked)

# --- Bloco de Inicialização da Interface ---
conversa_encerrada_flag = False; chat_session_conrad = None
if model_conrad and not initialization_error_message:
    try: chat_session_conrad = model_conrad.chat
    except Exception as e:
        err_detail = html.escape(str(e)); pt_msg, en_msg = SYS_MSG_CHAT_INIT_FAIL_PT, SYS_MSG_CHAT_INIT_FAIL_EN
        error_chat_init = f"{pt_msg}<br>{en_msg}<br><br>PT-BR: Detalhe: {err_detail}<br>ENG: Detail: {err_detail}"
        initialization_error_message = f"{initialization_error_message}<br><br>{error_chat_init}" if initialization_error_message else error_chat_init
        print(f"{pt_msg}\n{en_msg}\nPT-BR: Detalhe: {err_detail}\nENG: Detail: {err_detail}"); chat_session_conrad = None
elif not initialization_error_message and not model_conrad:
     initialization_error_message = f"{SYS_MSG_MODEL_INIT_FAIL_PT}<br>(Verifique se 'system_instruction_conrad' foi definida e se o modelo foi inicializado corretamente)."

if chat_session_conrad is not None and not initialization_error_message:
    _reset_chat_display_state(); user_input_area.disabled = send_button.disabled = False
    input_box.layout.display = 'flex'; status_message_area.value = ""
    display(IPyHTML("""
         <style>
            .chat-message-user-text, .chat-message-system-text {
                overflow-wrap: break-word; word-wrap: break-word; max-width: 100%;
            }
            .conrad-response-item {
                display: flex;
                flex-direction: column;
                overflow-wrap: break-word;
                word-wrap: break-word;
                max-width: 100%;
            }
            .conrad-response-header {
                display: flex;
                align-items: center;
                margin-bottom: 2px; /* Espaçamento Reduzido */
            }
            .conrad-avatar {
                margin-right: 5px; /* Espaçamento Ajustado */
                display: flex;
                align-items: center;
            }
            .conrad-avatar-icon {
                color: #e0e0e0;
            }
            .conrad-response-content {
                padding-left: 27px; /* (22px da cartola + 5px de margem) */
            }
            .conrad-response-content > *:first-child {
                margin-top: 0; /* Remove margem superior do primeiro elemento da resposta */
            }
            .codehilite {
                position: relative; background: #272822 !important; color: #f8f8f2 !important;
                padding: 10px !important; padding-bottom: 45px !important; border-radius: 5px !important;
                margin: 0px 0px 10px 0px !important; overflow-x: auto !important;
            }
            .codehilite pre {
                margin: 0 !important; padding: 0 !important; white-space: pre !important;
                background: transparent !important; border: none !important;
            }
            .code-buttons-container {
                position: absolute; bottom: 8px; left: 8px; display: flex; gap: 5px; z-index: 10;
            }
            .botao-code-action {
                width: 30px; height: 30px; display: flex; align-items: center; justify-content: center;
                background-color: rgba(200, 200, 200, 0.15); border: 1px solid rgba(200, 200, 200, 0.3);
                border-radius: 4px; cursor: pointer; opacity: 0.7;
                transition: opacity 0.2s ease-in-out, background-color 0.2s ease-in-out;
            }
            .botao-code-action:hover { opacity: 1; background-color: rgba(200, 200, 200, 0.3); }
            .botao-code-action svg { width: 16px; height: 16px; fill: #e0e0e0; }
            .botao-code-action:hover svg { fill: #ffffff; }
            .copiado-popup {
                position: absolute; bottom: 8px; left: 75px; background-color: #333; color: #fff;
                padding: 6px 12px; min-width: 100px; text-align: center; border-radius: 4px;
                font-size: 12px; z-index: 20; opacity: 0; transition: opacity 0.3s ease-in-out;
                pointer-events: none;
            }
            .copiado-popup.visivel { opacity: 1; }
            .code-language-indicator {
                position: absolute; bottom: 8px; right: 8px; padding: 3px 8px; border-radius: 4px;
                font-size: 12px; font-family: sans-serif; font-weight: 500;
                text-transform: capitalize; pointer-events: none;
            }
            .chat-message-user-text { text-align: justify; }
            button.jupyter-button.p-Widget[title="Enviar mensagem"], .conrad-send-button button.jupyter-button { border-radius: 6px !important; }

            /* --- ESTILIZAÇÃO UNIFICADA DAS BARRAS DE ROLAGEM --- */
            .conrad-chat-history::-webkit-scrollbar,
            .conrad-user-input-textarea textarea::-webkit-scrollbar,
            .codehilite::-webkit-scrollbar { width: 12px; height: 12px; }

            .conrad-chat-history::-webkit-scrollbar-track,
            .conrad-user-input-textarea textarea::-webkit-scrollbar-track,
            .codehilite::-webkit-scrollbar-track { background: #111; border-radius: 6px; }

            .conrad-chat-history::-webkit-scrollbar-thumb,
            .conrad-user-input-textarea textarea::-webkit-scrollbar-thumb,
            .codehilite::-webkit-scrollbar-thumb {
                background-color: #888; border-radius: 6px; border: 3px solid #111;
            }
            .conrad-chat-history::-webkit-scrollbar-thumb:hover,
            .conrad-user-input-textarea textarea::-webkit-scrollbar-thumb:hover,
            .codehilite::-webkit-scrollbar-thumb:hover { background-color: #aaa; }

            .conrad-chat-history::-webkit-scrollbar-button,
            .conrad-user-input-textarea textarea::-webkit-scrollbar-button,
            .codehilite::-webkit-scrollbar-button { display: none; }
         </style>
    """))
    display(ui_container)
    js_interactive_features = f"""<script>
    const MAX_USER_INPUT_LENGTH_JS = {MAX_USER_INPUT_LENGTH};
    function makeTextareaAutoResizable(el, maxH) {{ if (!el) return; el.style.resize = 'none'; el.style.boxSizing = 'border-box'; el.adjustTextareaHeight = () => {{ el.style.height = 'auto'; let scH = el.scrollHeight; if (maxH && scH > maxH) {{ el.style.height = maxH + 'px'; el.style.overflowY = 'auto'; }} else {{ el.style.height = scH + 'px'; el.style.overflowY = 'hidden'; }} }}; el.addEventListener('input', el.adjustTextareaHeight); setTimeout(() => {{ if (el.adjustTextareaHeight) el.adjustTextareaHeight(); }}, 100); }}
    function setupConradEnter() {{ const ta = document.querySelector('.conrad-user-input-textarea textarea'); if (ta) {{ makeTextareaAutoResizable(ta, 84); if (!ta.hasAttribute('data-conrad-enter')) {{ ta.addEventListener('keydown', function(e) {{ if (e.key === 'Enter' && !e.shiftKey) {{ e.preventDefault(); const sb = document.querySelector('.jupyter-button[title="Enviar mensagem"]'); if (sb && !sb.disabled) {{ const ui = ta.value.trim(); if (ui) {{ google.colab.kernel.invokeFunction('notebook.handleConradUserSubmitFromEnter', [ui], {{}}); ta.value = ''; if (typeof ta.adjustTextareaHeight === 'function') ta.adjustTextareaHeight(); if (!ta.disabled) {{ ta.focus(); }} }} }} }} }}); ta.setAttribute('data-conrad-enter', 'true'); }} }} else {{ setTimeout(setupConradEnter, 250); }} }}
    function focusInitial() {{ const ta = document.querySelector('.conrad-user-input-textarea textarea'); if (ta && !ta.disabled) ta.focus(); else setTimeout(focusInitial, 250); }}

    function copiarCodigoParaClipboard(codeBlockPreId, buttonElement, popupId) {{
        const preElement = document.getElementById(codeBlockPreId);
        const popupElement = document.getElementById(popupId);

        if (preElement && popupElement) {{
            let codeToCopy = preElement.innerText;
            if (codeToCopy.endsWith('\\n')) {{
                codeToCopy = codeToCopy.slice(0, -1);
            }}
            navigator.clipboard.writeText(codeToCopy).then(function() {{
                const msgCopiado = buttonElement.getAttribute('data-msg-copiado') || 'Código copiado!';
                popupElement.innerText = msgCopiado;
                popupElement.classList.add('visivel');
                setTimeout(function() {{ popupElement.classList.remove('visivel'); }}, 2500);
            }}, function(err) {{
                console.error('Erro ao copiar código: ', err);
                popupElement.innerText = 'Falha ao copiar!';
                popupElement.classList.add('visivel');
                 setTimeout(function() {{ popupElement.classList.remove('visivel'); }}, 3000);
            }});
        }} else {{
            if(!preElement) console.error('Elemento <pre> de código não encontrado:', codeBlockPreId);
            if(!popupElement) console.error('Elemento popup não encontrado:', popupId);
        }}
    }}

    function baixarCodigoComoArquivo(codeBlockPreId, nomeArquivo, mimeType) {{
        const preElement = document.getElementById(codeBlockPreId);
        if (preElement) {{
            let codigo = preElement.innerText;
            if (codigo.endsWith('\\n')) {{
                codigo = codigo.slice(0, -1);
            }}
            const effectiveMimeType = mimeType || 'text/plain;charset=utf-8';
            const blob = new Blob([codigo], {{ type: effectiveMimeType }});
            const link = document.createElement('a');
            link.href = URL.createObjectURL(blob);
            link.download = nomeArquivo || 'conrad_codigo.txt';
            document.body.appendChild(link);
            link.click();
            document.body.removeChild(link);
            URL.revokeObjectURL(link.href);
        }} else {{
            console.error('Elemento <pre> de código para download não encontrado:', codeBlockPreId);
            alert('Erro ao tentar baixar o código: elemento não encontrado.');
        }}
    }}

    if (document.readyState === 'complete' || document.readyState === 'interactive') {{ setupConradEnter(); focusInitial(); }}
    else {{ window.addEventListener('DOMContentLoaded', () => {{ setupConradEnter(); focusInitial(); }}); }}
    </script>"""
    display(IPyHTML(js_interactive_features))
    _focus_user_input_area()
else:
    display_error_msg = initialization_error_message
    if not display_error_msg: display_error_msg = f"{SYS_MSG_CRITICAL_FAILURE_PT}<br>{SYS_MSG_CRITICAL_FAILURE_EN}"
    display(Markdown(f"<div style='color:red; font-weight:bold;'>{display_error_msg}</div>"))
    if hasattr(user_input_area, 'disabled'): user_input_area.disabled = True
    if hasattr(send_button, 'disabled'): send_button.disabled = True
    if hasattr(input_box, 'layout') and hasattr(input_box.layout, 'display'): input_box.layout.display = 'none'